- Ingest data in data lakehouse 
- perform data quality checks and transform the data as requested -- silver_clean
- apply changes to the customers_data -- silver

LIVE   --> To refer to a table already created in a DLT Pipe <br>
STREAM --> To read data from another streaming table as a stream we need to use STREAM else it will read all the data.

In [0]:
CREATE OR REFRESH STREAMING TABLE bronze_customers
  COMMENT 'Raw customers data ingested from source system'
  TBLPROPERTIES ('quality' = 'bronze') AS
SELECT
  *,
  _metadata.file_path AS file_path,
  current_timestamp() AS ingestion_timestamp
FROM
  cloud_files(
    '/Volumes/circuitbox/landing/operational_data/customers/',
    'json',
    map('cloudFiles.inferColumnTypes', 'true')
  );

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers_clean(
  CONSTRAINT valid_customer_id EXPECT(customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_customer_name EXPECT(customer_name IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_telephone EXPECT(len(telephone) >= 10),
  CONSTRAINT valid_email EXPECT(email IS NOT NULL),
  CONSTRAINT valid_date_of_birth EXPECT(date_of_birth >= '1920-01-01')
)
  COMMENT 'Cleaned customers data'
  TBLPROPERTIES ('quality' = 'silver') AS
SELECT
  customer_id,
  customer_name,
  CAST(date_of_birth as DATE) as date_of_birth,
  telephone,
  email,
  CAST(created_date AS DATE) created_date
FROM
  STREAM(LIVE.bronze_customers)

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers
COMMENT 'SCD TYPE 1 Customers Data'
TBLPROPERTIES ('quality' = 'silver') ;

as silver table is scd type 1 , updated will be overwritten rather than a new row in SCD Type 2

KEYS -- > customer_id <br>
SEQUENCE BY --> to determine latest record for each customer

In [0]:
APPLY CHANGES INTO silver_customers
FROM STREAM(LIVE.silver_customers_clean)
KEYS (customer_id)
SEQUENCE BY created_date
STORED AS SCD TYPE 1 ;